In [1]:
import os

# isort: off
import xarray as xr
import pint_xarray
import cf_xarray.units
from pint_xarray import unit_registry as ureg
# isort: on

from utils import (
    PLASTIC_SHAPE_FACTORS,
    PLASTIC_SIZES,
    RESULTS_DIR,
    powerlaw_compute_bin_edges,
    powerlaw_compute_mass,
)

ureg.default_format = "cf"

Introduction
------------

This notebook computes initial microplastic emissions factors and size distribution factors for the *main* simulation. The goal is to have each source emit about 1 Gg/yr distributed over the six tracer sizes following a power law particle number size distribution whose slope equals the median slope of all atmospheric microplastic fiber observations in MPsizeBase [@Sonke2025].

We compute two sets of emissions factors:

* Five base emissions factors that set the total mass emitted by each source
* Six size distribution factors that distribute the mass over the six tracer sizes

Compute base emissions factors
------------------------------

We start by computing base emissions factors that will give emissions of about 1 Gg/yr from each source. This value is arbitrarily chosen and will be scaled up or down when we constrain the outputs to match observations.

We start with the emissions factors used in the `HEMCO_Config.rc` files of the *alternate* simulation. We reduce the emissions factor for the ocean source by a factor of 1e-9 so that ocean microplastic emissions will be roughly the same order of magnitude as emissions from other sources. We multiply the emissions factors by 6 (the emissions factor is applied to each of the six tracer sizees) to get the total emissions factor for each source.

In [2]:
# Emissions factors of alternate simulation
# Note: ocean emission factor reduced by factor of 1e-9
ef_alt = xr.DataArray(
    [1e2 * 1e-9, 1e-22, 1e-16, 1e-19, 1e-4],
    coords={"source": ["ocean", "mmpw", "agricultural", "residential", "road"]},
)
ef_alt = 6 * ef_alt
ef_alt.to_dataframe("Net emission factors of alternate simulation")

,Net emission factors of alternate simulation
source,
ocean,6.000000e-07
mmpw,6.000000e-22
agricultural,6.000000e-16
residential,6.000000e-19
road,6.000000e-04


Ocean emissions of the 0.3 µm tracer, which are based on accumulation-mode sea salt emissions, are much lower than ocean emissions of the other tracers, which are based on coarse-mode sea salt aerosol emissions. We load the outputs of the *alternate* simulation and compute a PST1 scaling factor that increases ocean emissions of the 0.3 µm tracer to match the emissions of the other tracer sizes.

In [3]:
# Load outputs of alternate simulation
sim_path = os.path.join(RESULTS_DIR, "sim_alt.nc")
sim_alt = xr.open_dataset(sim_path, chunks="auto")

# Compute factors to give same emissions for all tracer sizes
emis = (
    (sim_alt["emission"].pint.quantify() * sim_alt["area"].pint.quantify())
    .sum(dim=["lat", "lon"])
    .pint.to("Gg/year")
    .pint.dequantify()
)
size_scales = emis.sel(size=70) / emis
size_scales.to_dataframe("Scales to emit same mass for each tracer").unstack("source")

Scales to emit same mass for each tracer                                \
source                                    ocean mmpw agricultural residential   
size                                                                            
0.3                                    60.52459  1.0          1.0         1.0   
2.5                                     1.00000  1.0          1.0         1.0   
7.0                                     1.00000  1.0          1.0         1.0   
15.0                                    1.00000  1.0          1.0         1.0   
35.0                                    1.00000  1.0          1.0         1.0   
70.0                                    1.00000  1.0          1.0         1.0   

             
source road  
size         
0.3     1.0  
2.5     1.0  
7.0     1.0  
15.0    1.0  
35.0    1.0  
70.0    1.0

We multiply ocean PST1 emissions by the PST1 scaling factor and compute the total emissions from each source including scaled-up emissions of the 0.3 µm ocean tracer.

In [4]:
emis_new = (size_scales * emis).sum(dim="size")
emis_new.to_dataframe(f"Adjusted emissions [{emis.units}]")

,Adjusted emissions [Gg yr-1]
source,
ocean,279.039917
mmpw,0.493078
agricultural,4.364661
residential,0.143774
road,81.157318


We then compute scaling factors that adjust total emissions from each source to approximately 1 Gg/year.

In [5]:
ef_scales = 1 / emis_new
ef_scales.to_dataframe("Scales to emit ~1 Gg/year from each source")

,Scales to emit ~1 Gg/year from each source
source,
ocean,0.003584
mmpw,2.028076
agricultural,0.229113
residential,6.955384
road,0.012322


We multiply the emissions factors of the *alternate* simulation by these scaling factors to get the base emissions factors for the *main* simulation's `HEMCO_Config.rc` files.

In [6]:
ef_new = ef_scales * ef_alt
ef_new.to_dataframe("Base emissions factors")

,Base emissions factors
source,
ocean,2.150230e-09
mmpw,1.216846e-21
agricultural,1.374677e-16
residential,4.173231e-18
road,7.393049e-06


Compute size distribution
-------------------------

We then compute size distribution factors that distribute emissions over six tracer sizes according to a power law particle number size distribution with parameter $\alpha$ = 2.8 (log-log slope = -2.8). This value is the median of all atmospheric microplastic fiber observations in MPsizeBase [@Sonke2025] (see [prep-mp-size-base.ipynb](./prep-mpsizebase.ipynb)).

We use the following assumptions to compute the fraction of total mass in the size bin represented by each tracer:

* Microplastics have a density of 1 g/cm^3^
* Microplastic aerosols are fragments whose shape is approximated by an ellipsoid with width = 0.68 length and height = 0.4 height [@Sonke2025]
* Each tracer size (0.3, 2.5, 7, 15, 35, 70 µm) corresponds to the volume-weighted mean size of a size bin
* The lower bound of the smallest size bin is 0.092 µm

These assumptions imply size bin edges of 0.092, 1.30, 5.27, 9.46, 24.9, 50.4, and 99.5 µm.

In [7]:
alpha = 2.8
shape_factor = PLASTIC_SHAPE_FACTORS["fragments"]
bin_edges = powerlaw_compute_bin_edges(
    xmin=0.092, vol_mean_sizes=PLASTIC_SIZES, alpha=alpha, figs=3
)
bin_edges

[0.092, 1.3, 5.27, 9.46, 24.9, 50.4, 99.5]

We compute the fraction of total mass in each size bin and set these as the size distribution factors in the *main* simulation's `HEMCO_Config.rc` files.

In [8]:
masses = powerlaw_compute_mass(
    xmin=bin_edges[:-1] * ureg("µm"),
    xmax=bin_edges[1:] * ureg("µm"),
    alpha=alpha,
    c=100 * ureg(f"particle µm{alpha - 1}"),
    shape_factor=shape_factor,
    density=ureg("g/cm3"),
)
masses = masses.magnitude
frac = masses / masses.sum()

size_factors = xr.Dataset(
    data_vars={
        "min [µm]": ("size", bin_edges[:-1]),
        "max [µm]": ("size", bin_edges[1:]),
        "Fraction of total mass": ("size", frac.round(5)),
    },
    coords={"size": sim_alt["size"]},
)
size_factors.to_dataframe().rename_axis("size [µm]")

,min [µm],max [µm],Fraction of total mass
size [µm],,,
0.3,0.092,1.30,0.00526
2.5,1.300,5.27,0.02395
7.0,5.270,9.46,0.02996
15.0,9.460,24.90,0.13034
35.0,24.900,50.40,0.25247
70.0,50.400,99.50,0.55802


Compare to @Evangelou2026
-------------------------

### Our size bins over roughly 5-100 µm

In [9]:
from utils import powerlaw_compute_number

sizes = PLASTIC_SIZES[2:]
bin_edges = powerlaw_compute_bin_edges(
    xmin=5.27, vol_mean_sizes=sizes, alpha=alpha, figs=3
)
alpha = 2.8

numbers = powerlaw_compute_number(
    xmin=bin_edges[:-1] * ureg("µm"),
    xmax=bin_edges[1:] * ureg("µm"),
    alpha=alpha,
    c=100 * ureg(f"particle µm{alpha - 1}"),
)
numbers = numbers.magnitude
number_frac = numbers / numbers.sum()

masses = powerlaw_compute_mass(
    xmin=bin_edges[:-1] * ureg("µm"),
    xmax=bin_edges[1:] * ureg("µm"),
    alpha=alpha,
    c=100 * ureg(f"particle µm{alpha - 1}"),
    shape_factor=shape_factor,
    density=ureg("g/cm3"),
)
masses = masses.magnitude
mass_frac = masses / masses.sum()

size_factors = xr.Dataset(
    data_vars={
        "min [µm]": ("size", bin_edges[:-1]),
        "max [µm]": ("size", bin_edges[1:]),
        "Particles": ("size", numbers),
        "% of particles": ("size", number_frac.round(3)),
        "Mass": ("size", masses),
        "% of mass": ("size", mass_frac.round(3)),
    },
    coords={"size": sizes},
)
size_factors.to_dataframe().rename_axis("size [µm]")

,min [µm],max [µm],Particles,% of particles,Mass,% of mass
size [µm],,,,,,
7,5.27,9.46,1.816104,0.654,60.362479,0.031
15,9.46,24.90,0.802585,0.289,262.574197,0.134
35,24.90,50.40,0.122537,0.044,508.626393,0.260
70,50.40,99.50,0.033820,0.012,1124.171496,0.575


### @Evangelou2026's size bins

In [10]:
bin_edges = [5, 10, 25, 50, 100]
sizes = [round((a * b) ** 0.5, 1) for a, b in zip(bin_edges[:-1], bin_edges[1:])]
alpha = 2.8

numbers = powerlaw_compute_number(
    xmin=bin_edges[:-1] * ureg("µm"),
    xmax=bin_edges[1:] * ureg("µm"),
    alpha=alpha,
    c=100 * ureg(f"particle µm{alpha - 1}"),
)
numbers = numbers.magnitude
number_frac = numbers / numbers.sum()

masses = powerlaw_compute_mass(
    xmin=bin_edges[:-1] * ureg("µm"),
    xmax=bin_edges[1:] * ureg("µm"),
    alpha=alpha,
    c=100 * ureg(f"particle µm{alpha - 1}"),
    shape_factor=shape_factor,
    density=ureg("g/cm3"),
)
masses = masses.magnitude
mass_frac = masses / masses.sum()

size_factors = xr.Dataset(
    data_vars={
        "min [µm]": ("size", bin_edges[:-1]),
        "max [µm]": ("size", bin_edges[1:]),
        "Particles": ("size", numbers),
        "% of particles": ("size", number_frac.round(3)),
        "Mass": ("size", masses),
        "% of mass": ("size", mass_frac.round(3)),
    },
    coords={"size": sizes},
)
size_factors.to_dataframe().rename_axis("size [µm]")

,min [µm],max [µm],Particles,% of particles,Mass,% of mass
size [µm],,,,,,
7.1,5,10,2.185570,0.716,72.232391,0.037
15.8,10,25,0.711283,0.233,256.174077,0.130
35.4,25,50,0.120620,0.040,498.305861,0.253
70.7,50,100,0.034639,0.011,1144.806247,0.581
